In [5]:
import polars as pl
import numpy as np
from pathlib import Path
from catboost import CatBoostRanker, Pool
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("/kaggle/input/datasets/fedorfesenko/train-interactions")
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")

DATA_DIR: /kaggle/input/datasets/fedorfesenko/train-interactions


In [6]:
for f in [
    "train_dataset.parquet",
    "val_dataset.parquet",
    "test_dataset.parquet",
    "df_test.parquet",
    "df_ranking_val.parquet",
    "df_ranking_train.parquet",
    "candgen_features_train_v2.parquet",
    "candgen_features_val_v2.parquet",
    "candgen_features_test_v2.parquet",
]:
    print(f"{f}: {(DATA_DIR / f).exists()}")

train_dataset.parquet: True
val_dataset.parquet: True
test_dataset.parquet: True
df_test.parquet: True
df_ranking_val.parquet: True
df_ranking_train.parquet: True
candgen_features_train_v2.parquet: True
candgen_features_val_v2.parquet: True
candgen_features_test_v2.parquet: True


In [7]:
df_train = pl.read_parquet(DATA_DIR / "train_dataset.parquet")
df_val = pl.read_parquet(DATA_DIR / "val_dataset.parquet")
test_dataset = pl.read_parquet(DATA_DIR / "test_dataset.parquet")
df_test_orig = pl.read_parquet(DATA_DIR / "df_test.parquet")
df_ranking_val = pl.read_parquet(DATA_DIR / "df_ranking_val.parquet")

candgen_train_v2 = pl.read_parquet(DATA_DIR / "candgen_features_train_v2.parquet")
candgen_val_v2 = pl.read_parquet(DATA_DIR / "candgen_features_val_v2.parquet")
candgen_test_v2 = pl.read_parquet(DATA_DIR / "candgen_features_test_v2.parquet")

print(f"df_train: {df_train.shape}")
print(f"df_val: {df_val.shape}")
print(f"test_dataset: {test_dataset.shape}")
print(f"df_test_orig: {df_test_orig.shape}")
print(f"df_ranking_val: {df_ranking_val.shape}")
print(f"candgen_train_v2: {candgen_train_v2.shape}")
print(f"candgen_val_v2: {candgen_val_v2.shape}")
print(f"candgen_test_v2: {candgen_test_v2.shape}")

df_train: (1810569, 74)
df_val: (323736, 74)
test_dataset: (292826, 72)
df_test_orig: (292826, 11)
df_ranking_val: (323736, 14)
candgen_train_v2: (15769472, 17)
candgen_val_v2: (323736, 17)
candgen_test_v2: (292826, 17)


In [13]:
key_cols = ["user_id", "product_id"]

candgen_train_v2 = candgen_train_v2.unique(subset=key_cols, keep="first")
print(f"candgen_train_v2 dedup: {candgen_train_v2.shape}")

candgen_train_v2 = candgen_train_v2.drop(["als_score", "v5_score", "v6_score"])

df_train_v5 = df_train.join(candgen_train_v2, on=key_cols, how="left")
print(f"df_train_v5 после join: {df_train_v5.shape}")

df_ranking_train = pl.read_parquet(DATA_DIR / "df_ranking_train.parquet")
cat_train = (
    df_ranking_train
    .select(["user_id", "product_id", "request_id", "product_category"])
    .unique(subset=["user_id", "product_id", "request_id"], keep="first")
)
df_train_v5 = df_train_v5.join(cat_train, on=["user_id", "product_id", "request_id"], how="left")
print(f"df_train_v5 после product_category: {df_train_v5.shape}")
print(f"product_category null: {df_train_v5['product_category'].null_count()}")

df_train_v5 = df_train_v5.with_columns([
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.hour().alias("hour_of_day"),
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.weekday().alias("day_of_week"),
])

print(f"\ndf_train_v5: {df_train_v5.shape}")
print(f"nulls в candgen:")
print(df_train_v5.select(["als_score", "v5_score", "v6_score", "als_cosine", "v5_cosine", "v6_cosine"]).null_count())

del df_ranking_train, cat_train
import gc; gc.collect()

candgen_train_v2 dedup: (3372775, 17)
df_train_v5 после join: (1810569, 86)
df_train_v5 после product_category: (1810569, 87)
product_category null: 0

df_train_v5: (1810569, 87)
nulls в candgen:
shape: (1, 6)
┌───────────┬──────────┬──────────┬────────────┬───────────┬───────────┐
│ als_score ┆ v5_score ┆ v6_score ┆ als_cosine ┆ v5_cosine ┆ v6_cosine │
│ ---       ┆ ---      ┆ ---      ┆ ---        ┆ ---       ┆ ---       │
│ u32       ┆ u32      ┆ u32      ┆ u32        ┆ u32       ┆ u32       │
╞═══════════╪══════════╪══════════╪════════════╪═══════════╪═══════════╡
│ 0         ┆ 0        ┆ 0        ┆ 0          ┆ 0         ┆ 0         │
└───────────┴──────────┴──────────┴────────────┴───────────┴───────────┘


0

In [14]:
candgen_val_v2 = candgen_val_v2.unique(subset=key_cols, keep="first")
print(f"candgen_val_v2 dedup: {candgen_val_v2.shape}")

candgen_val_v2 = candgen_val_v2.drop(["als_score", "v5_score", "v6_score"])

df_val_v5 = df_val.join(candgen_val_v2, on=key_cols, how="left")
print(f"df_val_v5 после join: {df_val_v5.shape}")

if "product_category" not in df_val_v5.columns:
    cat_val = (
        df_ranking_val
        .select(["user_id", "product_id", "request_id", "product_category"])
        .unique(subset=["user_id", "product_id", "request_id"], keep="first")
    )
    df_val_v5 = df_val_v5.join(cat_val, on=["user_id", "product_id", "request_id"], how="left")
    print("Added product_category to val")

print(f"df_val_v5: {df_val_v5.shape}")
print(f"nulls в candgen:")
print(df_val_v5.select(["als_score", "v5_score", "v6_score", "als_cosine", "v5_cosine", "v6_cosine"]).null_count())
print(f"product_category null: {df_val_v5['product_category'].null_count()}")

candgen_val_v2 dedup: (175300, 17)
df_val_v5 после join: (323736, 86)
Added product_category to val
df_val_v5: (323736, 87)
nulls в candgen:
shape: (1, 6)
┌───────────┬──────────┬──────────┬────────────┬───────────┬───────────┐
│ als_score ┆ v5_score ┆ v6_score ┆ als_cosine ┆ v5_cosine ┆ v6_cosine │
│ ---       ┆ ---      ┆ ---      ┆ ---        ┆ ---       ┆ ---       │
│ u32       ┆ u32      ┆ u32      ┆ u32        ┆ u32       ┆ u32       │
╞═══════════╪══════════╪══════════╪════════════╪═══════════╪═══════════╡
│ 0         ┆ 0        ┆ 0        ┆ 0          ┆ 0         ┆ 0         │
└───────────┴──────────┴──────────┴────────────┴───────────┴───────────┘
product_category null: 0


In [18]:
key_cols_3 = ["user_id", "product_id", "request_id"]

candgen_test_v2 = pl.read_parquet(DATA_DIR / "candgen_features_test_v2.parquet")
candgen_test_v2 = candgen_test_v2.unique(subset=key_cols, keep="first")
print(f"candgen_test_v2 после перезагрузки: {candgen_test_v2.shape}")

candgen_test_v2 = candgen_test_v2.drop(["als_score", "v5_score", "v6_score"])
print(f"candgen_test_v2 после drop: {candgen_test_v2.shape}")

test_features_dedup = test_dataset.unique(subset=key_cols_3, keep="first")
cols_to_drop = [c for c in ["index", "timestamp", "source_type", "city_name"] if c in test_features_dedup.columns]
test_features_dedup = test_features_dedup.drop(cols_to_drop)
print(f"test_features_dedup: {test_features_dedup.shape}")
print(f"als_score in test_features_dedup: {'als_score' in test_features_dedup.columns}")

df_test_v5 = df_test_orig.join(test_features_dedup, on=key_cols_3, how="left", suffix="_features")

drop_right = [c for c in df_test_v5.columns if c.endswith("_right") or c.endswith("_features")]
if drop_right:
    df_test_v5 = df_test_v5.drop(drop_right)
    print(f"Dropped: {drop_right}")

df_test_v5 = df_test_v5.join(candgen_test_v2, on=key_cols, how="left")

if "hour_of_day" not in df_test_v5.columns:
    df_test_v5 = df_test_v5.with_columns([
        pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.hour().alias("hour_of_day"),
        pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.weekday().alias("day_of_week"),
    ])

print(f"\ndf_test_v5: {df_test_v5.shape}")
print(f"unique index: {df_test_v5['index'].n_unique()}")
print(f"als_score in df_test_v5: {'als_score' in df_test_v5.columns}")
print(f"columns: {df_test_v5.columns}")

candgen_test_v2 после перезагрузки: (148101, 17)
candgen_test_v2 после drop: (148101, 14)
test_features_dedup: (232166, 68)
als_score in test_features_dedup: True

df_test_v5: (292826, 88)
unique index: 292826
als_score in df_test_v5: True
columns: ['index', 'city_name', 'product_category', 'product_id', 'product_image', 'product_name', 'request_id', 'source_type', 'store_id', 'user_id', 'timestamp', 'n_views_all', 'n_clicks_all', 'n_cart_updates_all', 'n_purchases_all', 'n_views_30d', 'n_clicks_30d', 'n_cart_updates_30d', 'n_purchases_30d', 'n_views_7d', 'n_clicks_7d', 'n_cart_updates_7d', 'n_purchases_7d', 'n_views_1d', 'n_clicks_1d', 'n_cart_updates_1d', 'n_purchases_1d', 'days_since_first_action', 'days_since_last_action', 'item_n_views_all', 'item_n_clicks_all', 'item_n_cart_updates_all', 'item_n_purchases_all', 'item_n_views_30d', 'item_n_clicks_30d', 'item_n_cart_updates_30d', 'item_n_purchases_30d', 'item_n_views_7d', 'item_n_clicks_7d', 'item_n_cart_updates_7d', 'item_n_purcha

In [19]:
BASE_NUM = [
    "n_views_all", "n_clicks_all", "n_cart_updates_all", "n_purchases_all",
    "n_views_30d", "n_clicks_30d", "n_cart_updates_30d", "n_purchases_30d",
    "n_views_7d", "n_clicks_7d", "n_cart_updates_7d", "n_purchases_7d",
    "n_views_1d", "n_clicks_1d", "n_cart_updates_1d", "n_purchases_1d",
    "days_since_first_action", "days_since_last_action",
    "item_n_views_all", "item_n_clicks_all", "item_n_cart_updates_all", "item_n_purchases_all",
    "item_n_views_30d", "item_n_clicks_30d", "item_n_cart_updates_30d", "item_n_purchases_30d",
    "item_n_views_7d", "item_n_clicks_7d", "item_n_cart_updates_7d", "item_n_purchases_7d",
    "item_n_views_1d", "item_n_clicks_1d", "item_n_cart_updates_1d", "item_n_purchases_1d",
    "item_n_unique_users_all", "item_n_unique_users_30d", "item_n_unique_users_7d", "item_n_unique_users_1d",
    "item_days_since_first_view", "item_days_since_last_view",
    "ui_n_views_all", "ui_n_clicks_all", "ui_n_cart_updates_all", "ui_n_purchases_all",
    "ui_days_since_last_action", "ui_days_since_last_cart_update", "ui_days_since_last_purchase",
    "uc_n_views_all", "uc_n_clicks_all", "uc_n_cart_updates_all", "uc_n_purchases_all",
    "uc_n_views_30d", "uc_n_clicks_30d", "uc_n_cart_updates_30d", "uc_n_purchases_30d",
    "uc_n_views_7d", "uc_n_clicks_7d", "uc_n_cart_updates_7d", "uc_n_purchases_7d",
    "uc_days_since_last_purchase",
    "als_score", "v5_score", "v6_score",
    "hour_of_day", "day_of_week",
    "als_user_norm", "als_item_norm", "als_cosine", "als_norms_prod",
    "v5_user_norm",  "v5_item_norm",  "v5_cosine",  "v5_norms_prod",
    "v6_user_norm",  "v6_item_norm",  "v6_cosine",  "v6_norms_prod",
]
CAT_FEATURES = ["source_type", "city_name", "product_category"]
FEATURES = BASE_NUM + CAT_FEATURES

print(f"NUM: {len(BASE_NUM)}, CAT: {len(CAT_FEATURES)}, total: {len(FEATURES)}")

for name, df in [("train", df_train_v5), ("val", df_val_v5), ("test", df_test_v5)]:
    missing = [f for f in FEATURES if f not in df.columns]
    print(f"missing in {name}: {missing}")

def prepare_pool(df, features, cat_features, target_col="target", group_col="request_id"):
    df = df.sort(group_col)
    X = df.select(features).to_pandas()
    for c in features:
        if c not in cat_features:
            X[c] = X[c].astype("float32")
    for c in cat_features:
        X[c] = X[c].fillna("unknown").astype(str)
    if target_col in df.columns:
        y = df[target_col].to_numpy().astype("float32")
    else:
        y = np.zeros(len(df), dtype="float32")
    group_id = df[group_col].to_numpy()
    return Pool(data=X, label=y, group_id=group_id, cat_features=cat_features), df

print("\nPreparing pools...")
train_pool_v5, df_train_sorted_v5 = prepare_pool(df_train_v5, FEATURES, CAT_FEATURES)
val_pool_v5, df_val_sorted_v5 = prepare_pool(df_val_v5, FEATURES, CAT_FEATURES)
test_pool_v5, df_test_sorted_v5 = prepare_pool(df_test_v5, FEATURES, CAT_FEATURES)
print(f"train: {train_pool_v5.num_row()}")
print(f"val: {val_pool_v5.num_row()}")
print(f"test: {test_pool_v5.num_row()}")

NUM: 77, CAT: 3, total: 80
missing in train: []
missing in val: []
missing in test: []

Preparing pools...
train: 1810569
val: 323736
test: 292826


In [20]:
model_v5 = CatBoostRanker(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    task_type="CPU",
    thread_count=-1,
    verbose=100,
    random_seed=42,
    allow_writing_files=False,
)

model_v5.fit(
    train_pool_v5,
    eval_set=val_pool_v5,
    early_stopping_rounds=50,
    use_best_model=True,
    verbose=100,
)

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.8471696	best: 0.8471696 (0)	total: 4.81s	remaining: 1h 20m 1s
100:	test: 0.8817311	best: 0.8817311 (100)	total: 6m 56s	remaining: 1h 1m 46s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8824637603
bestIteration = 138

Shrink model to first 139 iterations.


CatBoostRanker(allow_writing_files=False, depth=8, eval_metric='NDCG:top=10', iterations=1000, learning_rate=0.05, loss_function='YetiRank', random_seed=42, task_type='CPU', verbose=100)

In [21]:
imp_v5 = model_v5.get_feature_importance(data=train_pool_v5, type="FeatureImportance")
imp_df_v5 = pl.DataFrame({
    "feature": model_v5.feature_names_,
    "importance": imp_v5,
}).sort("importance", descending=True)

print("=== Top-30 feature importance (v5) ===")
print(imp_df_v5.head(30))

print("\n=== Новые фичи (нормы, косинусы, произведения) ===")
new_feats = [
    "als_user_norm", "als_item_norm", "als_cosine", "als_norms_prod",
    "v5_user_norm", "v5_item_norm", "v5_cosine", "v5_norms_prod",
    "v6_user_norm", "v6_item_norm", "v6_cosine", "v6_norms_prod",
]
new_imp = imp_df_v5.filter(pl.col("feature").is_in(new_feats))
print(new_imp)

print(f"\nСуммарная важность новых фич: {new_imp['importance'].sum():.4f}")
print(f"Суммарная важность всех фич: {imp_df_v5['importance'].sum():.4f}")

=== Top-30 feature importance (v5) ===
shape: (30, 2)
┌────────────────────────┬────────────┐
│ feature                ┆ importance │
│ ---                    ┆ ---        │
│ str                    ┆ f64        │
╞════════════════════════╪════════════╡
│ v6_score               ┆ 0.065319   │
│ als_score              ┆ 0.032488   │
│ source_type            ┆ 0.020947   │
│ als_cosine             ┆ 0.018712   │
│ product_category       ┆ 0.012013   │
│ …                      ┆ …          │
│ uc_n_views_30d         ┆ 0.000106   │
│ uc_n_views_all         ┆ 0.000087   │
│ item_n_purchases_7d    ┆ 0.000085   │
│ n_purchases_7d         ┆ 0.000066   │
│ item_n_cart_updates_7d ┆ 0.000059   │
└────────────────────────┴────────────┘

=== Новые фичи (нормы, косинусы, произведения) ===
shape: (12, 2)
┌────────────────┬────────────┐
│ feature        ┆ importance │
│ ---            ┆ ---        │
│ str            ┆ f64        │
╞════════════════╪════════════╡
│ als_cosine     ┆ 0.018712   │
│ v6_co

In [22]:
test_preds_v5 = model_v5.predict(test_pool_v5)
print(f"test_preds_v5: {test_preds_v5.shape}")

df_sub_v5 = pl.DataFrame({
    "index": df_test_sorted_v5["index"].to_numpy(),
    "request_id": df_test_sorted_v5["request_id"].to_numpy(),
    "score": test_preds_v5.astype("float64"),
})
df_sub_v5_sorted = df_sub_v5.sort(["request_id", "score"], descending=[False, True])
df_sub_v5_final = df_sub_v5_sorted.select(["index", "request_id"])
df_sub_v5_final.write_csv("submission_v5.csv")

print(f"\nsubmission_v5.csv: {df_sub_v5_final.shape}")
print(f"unique index: {df_sub_v5_final['index'].n_unique()}")

test_preds_v5: (292826,)

submission_v5.csv: (292826, 2)
unique index: 292826


In [23]:
user_mean_target = df_train_v5.group_by("user_id").agg(
    pl.col("target").mean().alias("user_mean_target")
)
product_mean_target = df_train_v5.group_by("product_id").agg(
    pl.col("target").mean().alias("product_mean_target")
)
category_mean_target = df_train_v5.group_by("product_category").agg(
    pl.col("target").mean().alias("category_mean_target")
)
user_category_mean_target = df_train_v5.group_by(["user_id", "product_category"]).agg(
    pl.col("target").mean().alias("user_category_mean_target")
)

df_train_v6 = df_train_v5.join(user_mean_target, on="user_id", how="left")
df_train_v6 = df_train_v6.join(product_mean_target, on="product_id", how="left")
df_train_v6 = df_train_v6.join(category_mean_target, on="product_category", how="left")
df_train_v6 = df_train_v6.join(user_category_mean_target, on=["user_id", "product_category"], how="left")

df_val_v6 = df_val_v5.join(user_mean_target, on="user_id", how="left")
df_val_v6 = df_val_v6.join(product_mean_target, on="product_id", how="left")
df_val_v6 = df_val_v6.join(category_mean_target, on="product_category", how="left")
df_val_v6 = df_val_v6.join(user_category_mean_target, on=["user_id", "product_category"], how="left")

df_test_v6 = df_test_v5.join(user_mean_target, on="user_id", how="left")
df_test_v6 = df_test_v6.join(product_mean_target, on="product_id", how="left")
df_test_v6 = df_test_v6.join(category_mean_target, on="product_category", how="left")
df_test_v6 = df_test_v6.join(user_category_mean_target, on=["user_id", "product_category"], how="left")

print(f"df_train_v6: {df_train_v6.shape}")
print(f"df_val_v6: {df_val_v6.shape}")
print(f"df_test_v6: {df_test_v6.shape}")

df_train_v6: (1810569, 91)
df_val_v6: (323736, 91)
df_test_v6: (292826, 92)


In [24]:
def add_interactions_v6(df):
    return df.with_columns([
        (pl.col("als_cosine") * pl.col("user_mean_target")).alias("als_cosine_x_user_mean"),
        (pl.col("als_cosine") * pl.col("v6_score")).alias("als_cosine_x_v6"),
        (pl.col("v6_score") * pl.col("product_mean_target")).alias("v6_x_product_mean"),
        (pl.col("als_score") * pl.col("v5_score")).alias("als_x_v5"),
        (pl.col("v5_score") * pl.col("v6_score")).alias("v5_x_v6"),
        (pl.col("als_score") * pl.col("user_mean_target")).alias("als_x_user_mean"),
    ])

df_train_v6 = add_interactions_v6(df_train_v6)
df_val_v6 = add_interactions_v6(df_val_v6)
df_test_v6 = add_interactions_v6(df_test_v6)
print("Added interactions")

Added interactions


In [25]:
FEATURES_V6 = FEATURES + [
    "user_mean_target", "product_mean_target", "category_mean_target", "user_category_mean_target",
    "als_cosine_x_user_mean", "als_cosine_x_v6", "v6_x_product_mean",
    "als_x_v5", "v5_x_v6", "als_x_user_mean",
]

DROP_FEATS = ["als_norms_prod", "v5_norms_prod", "v6_norms_prod", "als_item_norm", "v6_user_norm"]
FEATURES_V6 = [f for f in FEATURES_V6 if f not in DROP_FEATS]

print(f"FEATURES_V6: {len(FEATURES_V6)}")

FEATURES_V6: 85


In [26]:
train_pool_v6, _ = prepare_pool(df_train_v6, FEATURES_V6, CAT_FEATURES)
val_pool_v6, _ = prepare_pool(df_val_v6, FEATURES_V6, CAT_FEATURES)
test_pool_v6, df_test_sorted_v6 = prepare_pool(df_test_v6, FEATURES_V6, CAT_FEATURES)

model_v6 = CatBoostRanker(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    task_type="CPU",
    thread_count=-1,
    verbose=100,
    random_seed=42,
    allow_writing_files=False,
)
model_v6.fit(train_pool_v6, eval_set=val_pool_v6, early_stopping_rounds=50, use_best_model=True, verbose=100)

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.8490800	best: 0.8490800 (0)	total: 4.56s	remaining: 1h 15m 54s
100:	test: 0.8817949	best: 0.8817949 (100)	total: 7m 15s	remaining: 1h 4m 34s
200:	test: 0.8834068	best: 0.8834476 (198)	total: 14m	remaining: 55m 40s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8835777597
bestIteration = 205

Shrink model to first 206 iterations.


CatBoostRanker(allow_writing_files=False, depth=8, eval_metric='NDCG:top=10', iterations=1000, learning_rate=0.05, loss_function='YetiRank', random_seed=42, task_type='CPU', verbose=100)

In [27]:
test_preds_v6 = model_v6.predict(test_pool_v6)
df_sub_v6 = pl.DataFrame({
    "index": df_test_sorted_v6["index"].to_numpy(),
    "request_id": df_test_sorted_v6["request_id"].to_numpy(),
    "score": test_preds_v6.astype("float64"),
})
df_sub_v6_sorted = df_sub_v6.sort(["request_id", "score"], descending=[False, True])
df_sub_v6_final = df_sub_v6_sorted.select(["index", "request_id"])
df_sub_v6_final.write_csv("submission_v6.csv")
print(f"submission_v6.csv: {df_sub_v6_final.shape}")
print(f"unique index: {df_sub_v6_final['index'].n_unique()}")

submission_v6.csv: (292826, 2)
unique index: 292826
